In [ ]:

import torch
import torchvision
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA is available:", torch.cuda.is_available())
import sys
!{sys.executable} -m pip install opencv-python matplotlib scikit-learn

In [ ]:
# once per runtime
!git clone -b feat/agent_tools https://github.com/flowithin/sam3.git
%cd sam3
!git pull
!pip install -e .

### login to Huggingface

In [ ]:
# Add huggingface token to the notebook secret keys
from google.colab import userdata
from huggingface_hub import login

# Get the token from Colab secrets
HF_TOKEN = userdata.get('HF_TOKEN')

# Log in to Hugging Face
if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("HF_TOKEN secret not found or notebook access not enabled.")

In [ ]:
from sam3.agent.agent_video_tracker import Sam3TrackingTool, DetectedObject, ObjectList

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
video_path = "/content/drive/MyDrive/Thomas pipelines/video assets/golden-state-warriors-minnesota-timberwolves-game-1-q1-05.32-05.25.mp4"
# !mkdir /content/annotated_basket

In [ ]:
sam3_tracker_players = Sam3TrackingTool(
    video_path=video_path,
    bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
)

In [ ]:
out = sam3_tracker_players._add_prompt(prompt_text_str="on court players")

In [ ]:
sam3_tracker_players._propagate()

In [ ]:
sam3_tracker_players._get_object_list().from_outputs_per_frame(sam3_tracker_players.outputs_per_frame)

### To save the checkpoints

In [ ]:
import os
os.makedirs("/content/checkpoints", exist_ok=True)
sam3_tracker_players._save_objects(path="/content/checkpoints")

In [ ]:
#now you can tarball it to pass it around
!tar -czf checkpoints.tar.gz /content/checkpoints/
#to untar
!tar -xzf checkpoints.tar.gz

### To load the checkpoints

In [ ]:
#to load object from checkpoints:
players = ObjectList()
for i in range(10):
  players.add_object(DetectedObject.load("/content/checkpoints/", id=i))
print(players)

now players is a class wrapping around list of Objects.

In [ ]:
#the mask
players.get_objects()[0].masks

In [ ]:
#the bounding boxes
players.get_objects()[0].bounding_boxes

### You can directly use some position decoding functions provided

In [ ]:
# the player with id=1 is defending player with id=2
obj1 = players.get_objects()[1]
obj2 = players.get_objects()[2]
obj1.near(frame_idx=0, object=obj1, radius=0.2)